In [21]:
# CELL 1: Locate the available TIRTL Traffic Count files

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Main folder where the downloaded datasets are stored
base_folder = Path(
    r"C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After"
)

# Find all folders containing TIRTL data
tirtl_folders = sorted([
    folder
    for folder in base_folder.iterdir()
    if folder.is_dir() and "tirtl" in folder.name.lower()
])

print("=" * 70)
print("TIRTL DATA FOLDERS")
print("=" * 70)

print("\nNumber of TIRTL folders found:", len(tirtl_folders))

for folder in tirtl_folders:
    print("-", folder.name)


# Search all TIRTL folders for CSV files
tirtl_files = []

for folder in tirtl_folders:
    tirtl_files.extend(folder.rglob("*.csv"))

tirtl_files = sorted(tirtl_files)


print("\n" + "=" * 70)
print("TIRTL CSV FILES")
print("=" * 70)

print("\nTotal CSV files found:", len(tirtl_files))

print("\nFirst 10 CSV files:")

for file in tirtl_files[:10]:
    print(file)

print("\nLast 5 CSV files:")

for file in tirtl_files[-5:]:
    print(file)

TIRTL DATA FOLDERS

Number of TIRTL folders found: 8
- tirtl_15min_volume_classification_april_2026
- tirtl_15min_volume_classification_august_2026 (1)
- tirtl_15min_volume_classification_february_2026
- tirtl_15min_volume_classification_january_2026
- tirtl_15min_volume_classification_july_2026
- tirtl_15min_volume_classification_june_2026
- tirtl_15min_volume_classification_march_2026
- tirtl_15min_volume_classification_may_2026

TIRTL CSV FILES

Total CSV files found: 215

First 10 CSV files:
C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After\tirtl_15min_volume_classification_april_2026\TIRTLDATA_20260401.csv
C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After\tirtl_15min_volume_classification_april_2026\TIRTLDATA_20260402.csv
C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After\tirtl_15min_volume_classification_april_2026\TIRTLDATA_20260403.csv
C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After\tirtl_15min_volume_classification_april_2026\TIRTLDATA_20260404.csv
C:\Users\LENOVO

In [23]:
# CELL 2: Inspect the structure of one TIRTL Traffic Count file

# Select the first available CSV file
sample_file = tirtl_files[0]

print("=" * 70)
print("SAMPLE TIRTL FILE")
print("=" * 70)

print("\nFile being inspected:")
print(sample_file)


# Read one daily file
sample_tirtl = pd.read_csv(
    sample_file,
    low_memory=False
)


# Dataset dimensions
print("\nDATASET SIZE")

print("Rows:", f"{len(sample_tirtl):,}")
print("Columns:", len(sample_tirtl.columns))


# Column names
print("\nCOLUMN NAMES")

for number, column in enumerate(
    sample_tirtl.columns,
    start=1
):
    print(f"{number}. {column}")


# Preview the data
print("\nFIRST 5 ROWS")

display(sample_tirtl.head())


# Data types
print("\nDATA TYPES")

print(sample_tirtl.dtypes)


# Missing values
print("\nMISSING VALUES IN SAMPLE FILE")

missing_sample = pd.DataFrame({
    "Column": sample_tirtl.columns,
    "Missing_Values": sample_tirtl.isna().sum().values,
    "Missing_Percentage": (
        sample_tirtl.isna().sum().values /
        len(sample_tirtl) * 100
    ).round(4)
})

display(missing_sample)


# Show unique values for columns with a small number of categories
print("\nCATEGORICAL COLUMN CHECK")

for column in sample_tirtl.columns:
    
    unique_count = sample_tirtl[column].nunique(dropna=True)
    
    if unique_count <= 20:
        print(f"\n{column} ({unique_count} unique values):")
        print(sample_tirtl[column].dropna().unique())

SAMPLE TIRTL FILE

File being inspected:
C:\Users\LENOVO\Desktop\Project Urban\Data\Raw\After\tirtl_15min_volume_classification_april_2026\TIRTLDATA_20260401.csv

DATASET SIZE
Rows: 1,160,005
Columns: 7

COLUMN NAMES
1. date
2. time_bin
3. site
4. heading
5. vehicle_class
6. speed_bin
7. volume

FIRST 5 ROWS


,date,time_bin,site,heading,vehicle_class,speed_bin,volume
0,2026-04-01,0:00,2,W,10,70km/hr to < 75km/hr,1
1,2026-04-01,0:00,2,W,11,70km/hr to < 75km/hr,1
2,2026-04-01,0:00,2,W,9,75km/hr to < 80km/hr,1
3,2026-04-01,0:00,2,W,1,80km/hr to < 85km/hr,1
4,2026-04-01,0:00,2,W,3,80km/hr to < 85km/hr,1



DATA TYPES
date             object
time_bin         object
site              int64
heading          object
vehicle_class     int64
speed_bin        object
volume            int64
dtype: object

MISSING VALUES IN SAMPLE FILE


,Column,Missing_Values,Missing_Percentage
0,date,0,0.0
1,time_bin,0,0.0
2,site,0,0.0
3,heading,0,0.0
4,vehicle_class,0,0.0
5,speed_bin,0,0.0
6,volume,0,0.0



CATEGORICAL COLUMN CHECK

date (1 unique values):
['2026-04-01']

heading (4 unique values):
['W' 'E' 'N' 'S']

vehicle_class (15 unique values):
[10 11  9  1  3  2  5  8  0  4  6  7 13 14 12]


In [25]:
# CELL 3: Check file structure, coverage and basic traffic information

from collections import Counter

file_information = []

all_sites = set()
all_vehicle_classes = set()
all_headings = set()
all_speed_bins = set()

date_records = Counter()

total_rows = 0
total_volume = 0

earliest_date = None
latest_date = None

different_structures = set()

expected_columns = [
    "date",
    "time_bin",
    "site",
    "heading",
    "vehicle_class",
    "speed_bin",
    "volume"
]


for number, file in enumerate(tirtl_files, start=1):

    print(
        f"Processing file {number}/{len(tirtl_files)}:",
        file.name
    )

    file_rows = 0
    file_dates = set()

    # Read each large daily file in smaller pieces
    for chunk in pd.read_csv(
        file,
        chunksize=200_000,
        low_memory=False
    ):

        file_rows += len(chunk)
        total_rows += len(chunk)

        # Record the column structure
        different_structures.add(
            tuple(chunk.columns)
        )

        # Convert date safely
        dates = pd.to_datetime(
            chunk["date"],
            errors="coerce"
        )

        valid_dates = dates.dropna()

        if not valid_dates.empty:

            chunk_min = valid_dates.min()
            chunk_max = valid_dates.max()

            if earliest_date is None or chunk_min < earliest_date:
                earliest_date = chunk_min

            if latest_date is None or chunk_max > latest_date:
                latest_date = chunk_max

            file_dates.update(
                valid_dates.dt.date.unique()
            )

            date_counts = (
                valid_dates.dt.date.value_counts()
            )

            for date_value, count_value in date_counts.items():
                date_records[date_value] += int(count_value)

        # Collect unique site IDs
        all_sites.update(
            chunk["site"].dropna().unique()
        )

        # Collect vehicle classes
        all_vehicle_classes.update(
            chunk["vehicle_class"].dropna().unique()
        )

        # Collect headings
        all_headings.update(
            chunk["heading"].dropna().unique()
        )

        # Collect speed bins
        all_speed_bins.update(
            chunk["speed_bin"].dropna().unique()
        )

        # Add traffic volume
        total_volume += chunk["volume"].sum()

    file_information.append({
        "File": file.name,
        "Rows": file_rows,
        "Dates_In_File": len(file_dates)
    })


# Convert file information to a table
file_information_df = pd.DataFrame(
    file_information
)


print("\n" + "=" * 70)
print("TIRTL DATA COVERAGE")
print("=" * 70)

print("\nFILES AND STRUCTURE")

print("CSV files checked:", len(tirtl_files))
print("Different column structures:", len(different_structures))
print("Total rows:", f"{total_rows:,}")


print("\nDATE COVERAGE")

print("Earliest date:", earliest_date)
print("Latest date:", latest_date)
print("Unique dates represented:", len(date_records))


print("\nTRAFFIC MONITORING")

print("Unique sites:", len(all_sites))
print(
    "Site IDs:",
    sorted(all_sites)
)

print(
    "Headings:",
    sorted(all_headings)
)

print(
    "Vehicle class codes:",
    sorted(all_vehicle_classes)
)

print(
    "Number of speed bins:",
    len(all_speed_bins)
)

print(
    "Total recorded traffic volume:",
    f"{total_volume:,}"
)


print("\nFILE SIZE SUMMARY")

print(
    "Smallest file:",
    f"{file_information_df['Rows'].min():,}",
    "rows"
)

print(
    "Largest file:",
    f"{file_information_df['Rows'].max():,}",
    "rows"
)

print(
    "Average file size:",
    f"{file_information_df['Rows'].mean():,.0f}",
    "rows"
)


print("\nFIRST 10 FILES")

display(
    file_information_df.head(10)
)

Processing file 1/215: TIRTLDATA_20260401.csv
Processing file 2/215: TIRTLDATA_20260402.csv
Processing file 3/215: TIRTLDATA_20260403.csv
Processing file 4/215: TIRTLDATA_20260404.csv
Processing file 5/215: TIRTLDATA_20260405.csv
Processing file 6/215: TIRTLDATA_20260406.csv
Processing file 7/215: TIRTLDATA_20260407.csv
Processing file 8/215: TIRTLDATA_20260408.csv
Processing file 9/215: TIRTLDATA_20260409.csv
Processing file 10/215: TIRTLDATA_20260410.csv
Processing file 11/215: TIRTLDATA_20260411.csv
Processing file 12/215: TIRTLDATA_20260412.csv
Processing file 13/215: TIRTLDATA_20260413.csv
Processing file 14/215: TIRTLDATA_20260414.csv
Processing file 15/215: TIRTLDATA_20260415.csv
Processing file 16/215: TIRTLDATA_20260416.csv
Processing file 17/215: TIRTLDATA_20260417.csv
Processing file 18/215: TIRTLDATA_20260418.csv
Processing file 19/215: TIRTLDATA_20260419.csv
Processing file 20/215: TIRTLDATA_20260420.csv
Processing file 21/215: TIRTLDATA_20260421.csv
Processing file 22/215

,File,Rows,Dates_In_File
0,TIRTLDATA_20260401.csv,1160005,1
1,TIRTLDATA_20260402.csv,1184145,1
2,TIRTLDATA_20260403.csv,685037,1
3,TIRTLDATA_20260404.csv,679487,1
4,TIRTLDATA_20260405.csv,623460,1
5,TIRTLDATA_20260406.csv,728207,1
6,TIRTLDATA_20260407.csv,1034582,1
7,TIRTLDATA_20260408.csv,1030425,1
8,TIRTLDATA_20260409.csv,1068753,1
9,TIRTLDATA_20260410.csv,1057132,1


In [27]:
# CELL 4: Check data quality across the TIRTL Traffic Count files

from collections import Counter

# Counters for missing values
missing_values = Counter()

# Other quality checks
negative_volume = 0
zero_volume = 0
invalid_dates = 0
invalid_times = 0
unexpected_headings = 0

# Track possible duplicate rows
duplicate_rows = 0

# Track sites available on each date
sites_per_date = {}

# Track total volume by month
monthly_volume = Counter()

# Track total volume by heading
heading_volume = Counter()

# Track total volume by vehicle class
vehicle_class_volume = Counter()

rows_checked = 0

expected_headings = {"N", "S", "E", "W"}


for number, file in enumerate(tirtl_files, start=1):

    print(
        f"Checking file {number}/{len(tirtl_files)}:",
        file.name
    )

    for chunk in pd.read_csv(
        file,
        chunksize=200_000,
        low_memory=False
    ):

        rows_checked += len(chunk)

        # ---------------------------------------------------------
        # Missing values
        # ---------------------------------------------------------

        chunk_missing = chunk.isna().sum()

        for column, value in chunk_missing.items():
            missing_values[column] += int(value)


        # ---------------------------------------------------------
        # Volume checks
        # ---------------------------------------------------------

        numeric_volume = pd.to_numeric(
            chunk["volume"],
            errors="coerce"
        )

        negative_volume += int(
            (numeric_volume < 0).sum()
        )

        zero_volume += int(
            (numeric_volume == 0).sum()
        )


        # ---------------------------------------------------------
        # Date checks
        # ---------------------------------------------------------

        parsed_dates = pd.to_datetime(
            chunk["date"],
            errors="coerce"
        )

        invalid_dates += int(
            parsed_dates.isna().sum()
        )


        # ---------------------------------------------------------
        # Time-bin checks
        # ---------------------------------------------------------

        parsed_times = pd.to_datetime(
            chunk["time_bin"],
            format="%H:%M",
            errors="coerce"
        )

        invalid_times += int(
            parsed_times.isna().sum()
        )


        # ---------------------------------------------------------
        # Heading checks
        # ---------------------------------------------------------

        headings = set(
            chunk["heading"]
            .dropna()
            .astype(str)
            .unique()
        )

        invalid_heading_values = (
            headings - expected_headings
        )

        if invalid_heading_values:

            unexpected_headings += int(
                chunk["heading"]
                .astype(str)
                .isin(invalid_heading_values)
                .sum()
            )


        # ---------------------------------------------------------
        # Duplicate check within each chunk
        # ---------------------------------------------------------

        duplicate_rows += int(
            chunk.duplicated().sum()
        )


        # ---------------------------------------------------------
        # Sites available per date
        # ---------------------------------------------------------

        valid_date_rows = chunk.loc[
            parsed_dates.notna()
        ].copy()

        valid_date_rows["_parsed_date"] = (
            parsed_dates[
                parsed_dates.notna()
            ].dt.date.values
        )

        for date_value, group in valid_date_rows.groupby(
            "_parsed_date"
        ):

            if date_value not in sites_per_date:
                sites_per_date[date_value] = set()

            sites_per_date[date_value].update(
                group["site"]
                .dropna()
                .unique()
            )


        # ---------------------------------------------------------
        # Monthly traffic volume
        # ---------------------------------------------------------

        valid_month_mask = (
            parsed_dates.notna() &
            numeric_volume.notna()
        )

        months = parsed_dates[
            valid_month_mask
        ].dt.to_period("M").astype(str)

        month_values = (
            pd.DataFrame({
                "month": months.values,
                "volume": numeric_volume[
                    valid_month_mask
                ].values
            })
            .groupby("month")["volume"]
            .sum()
        )

        for month, value in month_values.items():
            monthly_volume[month] += int(value)


        # ---------------------------------------------------------
        # Traffic volume by heading
        # ---------------------------------------------------------

        heading_summary = (
            chunk.assign(
                _volume=numeric_volume
            )
            .groupby("heading")["_volume"]
            .sum()
        )

        for heading, value in heading_summary.items():

            if pd.notna(value):
                heading_volume[heading] += int(value)


        # ---------------------------------------------------------
        # Traffic volume by vehicle class
        # ---------------------------------------------------------

        class_summary = (
            chunk.assign(
                _volume=numeric_volume
            )
            .groupby("vehicle_class")["_volume"]
            .sum()
        )

        for vehicle_class, value in class_summary.items():

            if pd.notna(value):
                vehicle_class_volume[
                    vehicle_class
                ] += int(value)


# -------------------------------------------------------------
# Create missing-value summary
# -------------------------------------------------------------

missing_summary_tirtl = pd.DataFrame({
    "Column": expected_columns,
    "Missing_Values": [
        missing_values[column]
        for column in expected_columns
    ]
})

missing_summary_tirtl[
    "Missing_Percentage"
] = (
    missing_summary_tirtl[
        "Missing_Values"
    ] / rows_checked * 100
).round(4)


# -------------------------------------------------------------
# Site coverage table
# -------------------------------------------------------------

site_coverage_tirtl = pd.DataFrame([
    {
        "Date": date_value,
        "Unique_Sites": len(site_set)
    }
    for date_value, site_set
    in sorted(sites_per_date.items())
])


# -------------------------------------------------------------
# Monthly volume table
# -------------------------------------------------------------

monthly_volume_tirtl = pd.DataFrame(
    sorted(monthly_volume.items()),
    columns=[
        "Month",
        "Total_Volume"
    ]
)


# -------------------------------------------------------------
# Vehicle class table
# -------------------------------------------------------------

vehicle_class_tirtl = pd.DataFrame(
    sorted(vehicle_class_volume.items()),
    columns=[
        "Vehicle_Class",
        "Total_Volume"
    ]
)

vehicle_class_tirtl = (
    vehicle_class_tirtl
    .sort_values(
        "Total_Volume",
        ascending=False
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------
# Display results
# -------------------------------------------------------------

print("\n" + "=" * 70)
print("TIRTL DATA QUALITY")
print("=" * 70)

print("\nTotal rows checked:", f"{rows_checked:,}")


print("\nMISSING VALUES")

display(missing_summary_tirtl)


print("\nVOLUME CHECK")

print(
    "Negative volume values:",
    f"{negative_volume:,}"
)

print(
    "Zero volume values:",
    f"{zero_volume:,}"
)


print("\nDATE AND TIME CHECK")

print(
    "Invalid dates:",
    f"{invalid_dates:,}"
)

print(
    "Invalid time bins:",
    f"{invalid_times:,}"
)


print("\nHEADING CHECK")

print(
    "Rows with unexpected headings:",
    f"{unexpected_headings:,}"
)


print("\nDUPLICATE CHECK")

print(
    "Potential duplicate rows found within chunks:",
    f"{duplicate_rows:,}"
)


print("\nSITE COVERAGE")

print(
    "Minimum sites recorded on a day:",
    site_coverage_tirtl[
        "Unique_Sites"
    ].min()
)

print(
    "Maximum sites recorded on a day:",
    site_coverage_tirtl[
        "Unique_Sites"
    ].max()
)

print(
    "Average sites recorded per day:",
    round(
        site_coverage_tirtl[
            "Unique_Sites"
        ].mean(),
        2
    )
)


print("\nMONTHLY RECORDED TRAFFIC VOLUME")

display(monthly_volume_tirtl)


print("\nTRAFFIC VOLUME BY HEADING")

for heading, value in sorted(
    heading_volume.items()
):
    print(
        f"{heading}: {value:,}"
    )


print("\nTRAFFIC VOLUME BY VEHICLE CLASS CODE")

display(vehicle_class_tirtl)

Checking file 1/215: TIRTLDATA_20260401.csv
Checking file 2/215: TIRTLDATA_20260402.csv
Checking file 3/215: TIRTLDATA_20260403.csv
Checking file 4/215: TIRTLDATA_20260404.csv
Checking file 5/215: TIRTLDATA_20260405.csv
Checking file 6/215: TIRTLDATA_20260406.csv
Checking file 7/215: TIRTLDATA_20260407.csv
Checking file 8/215: TIRTLDATA_20260408.csv
Checking file 9/215: TIRTLDATA_20260409.csv
Checking file 10/215: TIRTLDATA_20260410.csv
Checking file 11/215: TIRTLDATA_20260411.csv
Checking file 12/215: TIRTLDATA_20260412.csv
Checking file 13/215: TIRTLDATA_20260413.csv
Checking file 14/215: TIRTLDATA_20260414.csv
Checking file 15/215: TIRTLDATA_20260415.csv
Checking file 16/215: TIRTLDATA_20260416.csv
Checking file 17/215: TIRTLDATA_20260417.csv
Checking file 18/215: TIRTLDATA_20260418.csv
Checking file 19/215: TIRTLDATA_20260419.csv
Checking file 20/215: TIRTLDATA_20260420.csv
Checking file 21/215: TIRTLDATA_20260421.csv
Checking file 22/215: TIRTLDATA_20260422.csv
Checking file 23/21

,Column,Missing_Values,Missing_Percentage
0,date,0,0.0
1,time_bin,0,0.0
2,site,0,0.0
3,heading,0,0.0
4,vehicle_class,0,0.0
5,speed_bin,0,0.0
6,volume,0,0.0



VOLUME CHECK
Negative volume values: 0
Zero volume values: 0

DATE AND TIME CHECK
Invalid dates: 0
Invalid time bins: 0

HEADING CHECK
Rows with unexpected headings: 0

DUPLICATE CHECK
Potential duplicate rows found within chunks: 0

SITE COVERAGE
Minimum sites recorded on a day: 275
Maximum sites recorded on a day: 292
Average sites recorded per day: 286.28

MONTHLY RECORDED TRAFFIC VOLUME


,Month,Total_Volume
0,2026-01,514997294
1,2026-02,496601945
2,2026-03,527082717
3,2026-04,480822073
4,2026-05,512161106
5,2026-06,500705916
6,2026-07,496488261
7,2026-08,82213377



TRAFFIC VOLUME BY HEADING
E: 1,631,313,339
N: 192,587,961
S: 185,486,942
W: 1,601,684,447

TRAFFIC VOLUME BY VEHICLE CLASS CODE


,Vehicle_Class,Total_Volume
0,1,3019852979
1,3,355039100
2,9,61301304
3,4,38588541
4,10,38210124
5,2,34354259
6,0,14357498
7,7,12699106
8,5,12188849
9,8,9502705


In [29]:
# CELL 5: Create hourly traffic summaries and save current analysis outputs

from pathlib import Path
from collections import Counter

# -------------------------------------------------------------
# Containers for aggregated results
# -------------------------------------------------------------

hourly_volume = Counter()
daily_volume = Counter()
hour_of_day_volume = Counter()
hour_of_day_intervals = Counter()

site_volume = Counter()

rows_processed = 0


# -------------------------------------------------------------
# Process each file in manageable chunks
# -------------------------------------------------------------

for number, file in enumerate(tirtl_files, start=1):

    print(
        f"Aggregating file {number}/{len(tirtl_files)}:",
        file.name
    )

    for chunk in pd.read_csv(
        file,
        chunksize=200_000,
        low_memory=False
    ):

        rows_processed += len(chunk)

        # Convert date and time into one timestamp
        timestamp = pd.to_datetime(
            chunk["date"].astype(str)
            + " "
            + chunk["time_bin"].astype(str),
            errors="coerce"
        )

        volume = pd.to_numeric(
            chunk["volume"],
            errors="coerce"
        )

        valid = (
            timestamp.notna()
            & volume.notna()
        )

        temp = pd.DataFrame({
            "timestamp": timestamp[valid],
            "site": chunk.loc[valid, "site"].values,
            "volume": volume[valid].values
        })

        # ---------------------------------------------------------
        # Convert 15-minute observations to hourly periods
        # ---------------------------------------------------------

        temp["hour"] = (
            temp["timestamp"].dt.floor("h")
        )

        temp["date"] = (
            temp["timestamp"].dt.date
        )

        temp["hour_of_day"] = (
            temp["timestamp"].dt.hour
        )


        # ---------------------------------------------------------
        # Total volume for each hour
        # ---------------------------------------------------------

        hourly_chunk = (
            temp.groupby("hour")["volume"]
            .sum()
        )

        for hour, value in hourly_chunk.items():
            hourly_volume[hour] += int(value)


        # ---------------------------------------------------------
        # Daily traffic volume
        # ---------------------------------------------------------

        daily_chunk = (
            temp.groupby("date")["volume"]
            .sum()
        )

        for date_value, value in daily_chunk.items():
            daily_volume[date_value] += int(value)


        # ---------------------------------------------------------
        # Site traffic volume
        # ---------------------------------------------------------

        site_chunk = (
            temp.groupby("site")["volume"]
            .sum()
        )

        for site, value in site_chunk.items():
            site_volume[site] += int(value)


# -------------------------------------------------------------
# Create hourly dataframe
# -------------------------------------------------------------

hourly_tirtl = pd.DataFrame(
    sorted(hourly_volume.items()),
    columns=[
        "Hour",
        "Traffic_Volume"
    ]
)

hourly_tirtl["Hour_of_Day"] = (
    pd.to_datetime(
        hourly_tirtl["Hour"]
    ).dt.hour
)


# -------------------------------------------------------------
# Calculate average traffic by hour of day
# -------------------------------------------------------------

hourly_pattern_tirtl = (
    hourly_tirtl
    .groupby("Hour_of_Day")
    ["Traffic_Volume"]
    .agg(
        Total_Volume="sum",
        Number_of_Hours="count",
        Average_Hourly_Volume="mean"
    )
    .reset_index()
)

hourly_pattern_tirtl[
    "Average_Hourly_Volume"
] = (
    hourly_pattern_tirtl[
        "Average_Hourly_Volume"
    ].round(2)
)


# -------------------------------------------------------------
# Daily dataframe
# -------------------------------------------------------------

daily_tirtl = pd.DataFrame(
    sorted(daily_volume.items()),
    columns=[
        "Date",
        "Traffic_Volume"
    ]
)

daily_tirtl["Date"] = pd.to_datetime(
    daily_tirtl["Date"]
)

daily_tirtl["Day"] = (
    daily_tirtl["Date"]
    .dt.day_name()
)


# -------------------------------------------------------------
# Average daily volume by day of week
# -------------------------------------------------------------

day_pattern_tirtl = (
    daily_tirtl
    .groupby("Day")["Traffic_Volume"]
    .agg(
        Total_Volume="sum",
        Number_of_Days="count",
        Average_Daily_Volume="mean"
    )
    .reset_index()
)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_pattern_tirtl["Day"] = pd.Categorical(
    day_pattern_tirtl["Day"],
    categories=day_order,
    ordered=True
)

day_pattern_tirtl = (
    day_pattern_tirtl
    .sort_values("Day")
    .reset_index(drop=True)
)

day_pattern_tirtl[
    "Average_Daily_Volume"
] = (
    day_pattern_tirtl[
        "Average_Daily_Volume"
    ].round(2)
)


# -------------------------------------------------------------
# Site summary
# -------------------------------------------------------------

site_summary_tirtl = pd.DataFrame(
    sorted(site_volume.items()),
    columns=[
        "Site",
        "Total_Volume"
    ]
)

site_summary_tirtl = (
    site_summary_tirtl
    .sort_values(
        "Total_Volume",
        ascending=False
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------
# Display useful results
# -------------------------------------------------------------

print("\n" + "=" * 70)
print("CURRENT TIRTL TRAFFIC ANALYSIS")
print("=" * 70)

print(
    "\nRows processed:",
    f"{rows_processed:,}"
)

print(
    "Hourly periods created:",
    f"{len(hourly_tirtl):,}"
)

print(
    "Days represented:",
    f"{len(daily_tirtl):,}"
)


print("\nHOURLY TRAFFIC PATTERN")

display(hourly_pattern_tirtl)


print("\nDAY-OF-WEEK TRAFFIC PATTERN")

display(day_pattern_tirtl)


print("\nTOP 10 SITES BY RECORDED TRAFFIC VOLUME")

display(
    site_summary_tirtl.head(10)
)


# -------------------------------------------------------------
# Identify current peak periods
# -------------------------------------------------------------

peak_hour = hourly_pattern_tirtl.loc[
    hourly_pattern_tirtl[
        "Average_Hourly_Volume"
    ].idxmax()
]

peak_day = day_pattern_tirtl.loc[
    day_pattern_tirtl[
        "Average_Daily_Volume"
    ].idxmax()
]


print("\nCURRENT TRAFFIC PATTERNS")

print(
    "Highest average traffic hour:",
    f"{int(peak_hour['Hour_of_Day']):02d}:00",
    "-",
    f"{peak_hour['Average_Hourly_Volume']:,.2f}"
)

print(
    "Highest average traffic day:",
    peak_day["Day"],
    "-",
    f"{peak_day['Average_Daily_Volume']:,.2f}"
)


# -------------------------------------------------------------
# Save current outputs
# -------------------------------------------------------------

output_folder = Path(
    r"C:\Users\LENOVO\Desktop\Project Urban\Data\EDA_Output\TIRTL"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


hourly_tirtl.to_csv(
    output_folder / "hourly_traffic_volume.csv",
    index=False
)

hourly_pattern_tirtl.to_csv(
    output_folder / "hourly_traffic_pattern.csv",
    index=False
)

daily_tirtl.to_csv(
    output_folder / "daily_traffic_volume.csv",
    index=False
)

day_pattern_tirtl.to_csv(
    output_folder / "day_of_week_traffic_pattern.csv",
    index=False
)

site_summary_tirtl.to_csv(
    output_folder / "site_traffic_summary.csv",
    index=False
)

monthly_volume_tirtl.to_csv(
    output_folder / "monthly_traffic_volume.csv",
    index=False
)

missing_summary_tirtl.to_csv(
    output_folder / "missing_values_summary.csv",
    index=False
)

vehicle_class_tirtl.to_csv(
    output_folder / "vehicle_class_volume.csv",
    index=False
)


print("\nOUTPUT FILES CREATED")

for file in sorted(
    output_folder.glob("*.csv")
):
    print("-", file.name)


print(
    "\nCurrent TIRTL analysis outputs "
    "saved successfully."
)

Aggregating file 1/215: TIRTLDATA_20260401.csv
Aggregating file 2/215: TIRTLDATA_20260402.csv
Aggregating file 3/215: TIRTLDATA_20260403.csv
Aggregating file 4/215: TIRTLDATA_20260404.csv
Aggregating file 5/215: TIRTLDATA_20260405.csv
Aggregating file 6/215: TIRTLDATA_20260406.csv
Aggregating file 7/215: TIRTLDATA_20260407.csv
Aggregating file 8/215: TIRTLDATA_20260408.csv
Aggregating file 9/215: TIRTLDATA_20260409.csv
Aggregating file 10/215: TIRTLDATA_20260410.csv
Aggregating file 11/215: TIRTLDATA_20260411.csv
Aggregating file 12/215: TIRTLDATA_20260412.csv
Aggregating file 13/215: TIRTLDATA_20260413.csv
Aggregating file 14/215: TIRTLDATA_20260414.csv
Aggregating file 15/215: TIRTLDATA_20260415.csv
Aggregating file 16/215: TIRTLDATA_20260416.csv
Aggregating file 17/215: TIRTLDATA_20260417.csv
Aggregating file 18/215: TIRTLDATA_20260418.csv
Aggregating file 19/215: TIRTLDATA_20260419.csv
Aggregating file 20/215: TIRTLDATA_20260420.csv
Aggregating file 21/215: TIRTLDATA_20260421.csv
A

,Hour_of_Day,Total_Volume,Number_of_Hours,Average_Hourly_Volume
0,0,35762818,215,166338.69
1,1,21787382,215,101336.66
2,2,16438076,215,76456.17
3,3,19293269,215,89736.13
4,4,43248134,215,201154.11
5,5,120748542,215,561621.13
6,6,171052145,215,795591.37
7,7,193273147,215,898944.87
8,8,203646831,215,947194.56
9,9,203019042,215,944274.61



DAY-OF-WEEK TRAFFIC PATTERN


,Day,Total_Volume,Number_of_Days,Average_Daily_Volume
0,Monday,506559230,31,16340620.32
1,Tuesday,527546089,31,17017615.77
2,Wednesday,540957720,31,17450249.03
3,Thursday,528934773,30,17631159.10
4,Friday,556423133,30,18547437.77
5,Saturday,504861916,31,16285868.26
6,Sunday,445789828,31,14380317.03



TOP 10 SITES BY RECORDED TRAFFIC VOLUME


,Site,Total_Volume
0,26,22384495
1,24,22301230
2,167,22153668
3,173,22086659
4,580,22049437
5,16,21131815
6,166,21080859
7,25,21020677
8,18,20916322
9,179,20713598



CURRENT TRAFFIC PATTERNS
Highest average traffic hour: 16:00 - 1,188,707.63
Highest average traffic day: Friday - 18,547,437.77

OUTPUT FILES CREATED
- daily_traffic_volume.csv
- day_of_week_traffic_pattern.csv
- hourly_traffic_pattern.csv
- hourly_traffic_volume.csv
- missing_values_summary.csv
- monthly_traffic_volume.csv
- site_traffic_summary.csv
- vehicle_class_volume.csv

Current TIRTL analysis outputs saved successfully.
